# 03bis-B — Expériences Agent IA 2
## Comparaison vecteur / vision avec une référence SIG

Ce notebook est un **banc d'essais**.

Il ne remplace pas la méthode 0 et **n'écrit jamais** dans :

```text
data/output/terrain_forest_annotations.json
```

Il part exactement des données produites par `03bis-A`.

Architecture expérimentale :

```text
03bis-A — méthode 0
récupération OSM stable
        ↓
forest_osm_union.geojson
        ↓
┌────────────────────────────────────────────┐
│  RÉFÉRENCE SIG : INTERSECT                 │
│  LineString route ∩ polygones forêt        │
└────────────────────────────────────────────┘
        ↓
        ├─────────────────────────────┐
        │                             │
        ▼                             ▼
TEST A                          TEST B1
Agent IA vectoriel             Agent IA visuel
GeoJSON + trace                carte abstraite
                               route + polygones OSM

Plus tard :
TEST B2 — vraie reconnaissance visuelle de forêt
orthophoto/satellite + trace, sans polygone forêt déjà dessiné
```

### Terminologie retenue pour le PIR

- **Méthode 0** : baseline OSM + corridor, déjà fonctionnelle.
- **Référence SIG** : `INTERSECT`, utilisée uniquement pour mesurer les expériences.
- **Test A** : raisonnement spatial vectoriel par Agent IA.
- **Test B1** : raisonnement spatial visuel sur une carte dérivée d'OSM.
- **Test B2** : véritable reconnaissance visuelle de forêt sur imagerie réelle — à faire ensuite.

In [ ]:
from pathlib import Path
import base64
import json

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from pyproj import Transformer
from shapely.geometry import (
    LineString,
    Point,
    shape,
    mapping,
)
from shapely.ops import (
    transform,
)

## 1. Configuration des expériences

Les tests sont indépendants.

Pour commencer simplement :

```python
RUN_VECTOR_TEST = True
RUN_VISUAL_OSM_TEST = False
```

Puis activer le test visuel B1 lorsque le modèle multimodal est installé.

In [ ]:
ROUTE_PATH = Path(
    "data/output/route_resampled.csv"
)

FOREST_UNION_PATH = Path(
    "data/output/forest_osm_union.geojson"
)

FOREST_CONTEXT_PATH = Path(
    "data/output/forest_osm_context.json"
)

OUTPUT_DIR = Path("data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_VECTOR_TEST = True
RUN_VISUAL_OSM_TEST = False

OLLAMA_URL = "http://localhost:11434/api/chat"

TEXT_MODEL = "llama3.1:8b"
VISION_MODEL = "qwen3-vl:4b"

# Garde-fous seulement : ce ne sont pas des listes de choix.
VECTOR_SAMPLE_BOUNDS_M = (25.0, 300.0)
VECTOR_SIMPLIFY_BOUNDS_M = (5.0, 200.0)

VISION_MARKER_SPACING_M = 500.0
VISION_IMAGE_DPI = 160

## 2. Charger exactement le contexte produit par la méthode 0

Le notebook ne refait **aucune requête Overpass**.

In [ ]:
for path in [
    ROUTE_PATH,
    FOREST_UNION_PATH,
    FOREST_CONTEXT_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"{path} absent. "
            "Exécute d'abord 03_BIS_A_METHODE0_FOREST_OSM_STABLE.ipynb."
        )

route = pd.read_csv(ROUTE_PATH)

with FOREST_UNION_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    forest_wgs84 = shape(
        json.load(f)
    )

with FOREST_CONTEXT_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    forest_context = json.load(f)

ROUTE_LENGTH_M = float(
    route["distance_m"].iloc[-1]
)

print("Longueur route :", round(ROUTE_LENGTH_M, 1), "m")
print(
    "Couverture ponctuelle méthode 0 :",
    round(
        100
        * float(
            forest_context[
                "osm_point_coverage_ratio"
            ]
        ),
        1,
    ),
    "%",
)
print(
    "Tags :",
    forest_context["forest_tags"],
)

# Partie I — Référence SIG : `INTERSECT`

`INTERSECT` n'est pas un Agent IA.

Il fournit une **référence géométrique déterministe** pour quantifier les performances
des tests A et B1.

In [ ]:
def utm_epsg_for_lonlat(
    lon,
    lat,
):
    zone = int(
        (lon + 180.0) / 6.0
    ) + 1

    return (
        32600 + zone
        if lat >= 0
        else 32700 + zone
    )


center_lon = float(route["lon"].mean())
center_lat = float(route["lat"].mean())

utm_epsg = utm_epsg_for_lonlat(
    center_lon,
    center_lat,
)

to_metric = Transformer.from_crs(
    "EPSG:4326",
    f"EPSG:{utm_epsg}",
    always_xy=True,
)

to_wgs84 = Transformer.from_crs(
    f"EPSG:{utm_epsg}",
    "EPSG:4326",
    always_xy=True,
)

forest_m = transform(
    to_metric.transform,
    forest_wgs84,
)

route_distances = (
    route["distance_m"]
    .to_numpy(dtype=float)
)

projected_points = [
    to_metric.transform(
        float(lon),
        float(lat),
    )
    for lon, lat in zip(
        route["lon"],
        route["lat"],
    )
]

print("CRS métrique :", f"EPSG:{utm_epsg}")

In [ ]:
def extract_line_geometries(geometry):
    if geometry is None or geometry.is_empty:
        return []

    if geometry.geom_type == "LineString":
        return [geometry]

    if geometry.geom_type == "MultiLineString":
        return list(geometry.geoms)

    if geometry.geom_type == "GeometryCollection":
        lines = []

        for child in geometry.geoms:
            lines.extend(
                extract_line_geometries(
                    child
                )
            )

        return lines

    return []


def merge_intervals(
    intervals,
    gap_tolerance_m=1.0,
):
    if not intervals:
        return []

    clean = sorted(
        [
            [
                max(0.0, float(start)),
                min(
                    ROUTE_LENGTH_M,
                    float(end),
                ),
            ]
            for start, end in intervals
            if float(end) > float(start)
        ],
        key=lambda x: x[0],
    )

    if not clean:
        return []

    merged = [clean[0]]

    for start, end in clean[1:]:
        if (
            start
            <= merged[-1][1]
            + gap_tolerance_m
        ):
            merged[-1][1] = max(
                merged[-1][1],
                end,
            )
        else:
            merged.append(
                [start, end]
            )

    return [
        (
            float(start),
            float(end),
        )
        for start, end in merged
    ]


def compute_reference_intersect():
    intervals = []

    for i in range(len(route) - 1):
        segment = LineString([
            projected_points[i],
            projected_points[i + 1],
        ])

        if segment.length <= 0:
            continue

        intersection = (
            segment.intersection(
                forest_m
            )
        )

        for line in extract_line_geometries(
            intersection
        ):
            if line.length <= 0:
                continue

            p0 = Point(line.coords[0])
            p1 = Point(line.coords[-1])

            f0 = (
                segment.project(p0)
                / segment.length
            )

            f1 = (
                segment.project(p1)
                / segment.length
            )

            d0 = float(route_distances[i])
            d1 = float(route_distances[i + 1])

            start = (
                d0
                + min(f0, f1)
                * (d1 - d0)
            )

            end = (
                d0
                + max(f0, f1)
                * (d1 - d0)
            )

            intervals.append(
                (start, end)
            )

    return merge_intervals(
        intervals,
        gap_tolerance_m=1.0,
    )


reference_intervals = (
    compute_reference_intersect()
)

reference_length_m = sum(
    end - start
    for start, end in reference_intervals
)

print(
    "Référence SIG :",
    len(reference_intervals),
    "portion(s)",
)

print(
    "Longueur réellement intersectée :",
    round(reference_length_m, 1),
    "m",
)

print(
    "Part de l'itinéraire :",
    round(
        100
        * reference_length_m
        / max(ROUTE_LENGTH_M, 1e-9),
        1,
    ),
    "%",
)

display(
    pd.DataFrame(
        reference_intervals,
        columns=["start_m", "end_m"],
    )
)

# Partie II — Outils communs des Agents IA

In [ ]:
def extract_json_object(text):
    text = text.strip()

    try:
        return json.loads(text)

    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")

        if (
            start == -1
            or end == -1
            or end <= start
        ):
            raise

        return json.loads(
            text[start:end + 1]
        )


def ask_ollama_json(
    model,
    prompt,
    images_base64=None,
):
    message = {
        "role": "user",
        "content": prompt,
    }

    if images_base64:
        message["images"] = images_base64

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "messages": [message],
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0,
            },
        },
        timeout=180,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Ollama HTTP {response.status_code}: "
            f"{response.text}"
        )

    return extract_json_object(
        response.json()[
            "message"
        ]["content"]
    )


def normalize_agent_intervals(
    raw_intervals,
):
    clean = []

    for item in raw_intervals:
        try:
            start = float(item["start_m"])
            end = float(item["end_m"])

            confidence = float(
                item.get(
                    "confidence",
                    0.5,
                )
            )

        except (
            KeyError,
            TypeError,
            ValueError,
        ):
            continue

        start = max(
            0.0,
            min(start, ROUTE_LENGTH_M),
        )

        end = max(
            0.0,
            min(end, ROUTE_LENGTH_M),
        )

        if end <= start:
            continue

        clean.append({
            "start_m": start,
            "end_m": end,
            "confidence": min(
                1.0,
                max(0.0, confidence),
            ),
        })

    return sorted(
        clean,
        key=lambda x: x["start_m"],
    )

# Partie III — Test A : Agent IA vectoriel

### Question expérimentale

> Un Agent IA peut-il retrouver les portions forestières en raisonnant directement
> sur une trace échantillonnée et des polygones vectoriels ?

Le résultat `INTERSECT` n'est pas fourni à l'agent.

In [ ]:
def choose_vector_representation():
    prompt = f'''
Tu dois choisir la granularité d'une représentation vectorielle
pour analyser spatialement un itinéraire.

Longueur de l'itinéraire :
{ROUTE_LENGTH_M:.1f} m

Contexte OSM brut :
{json.dumps(forest_context, ensure_ascii=False)}

Choisis librement :
- route_sample_spacing_m entre {VECTOR_SAMPLE_BOUNDS_M[0]} et {VECTOR_SAMPLE_BOUNDS_M[1]} m ;
- polygon_simplify_m entre {VECTOR_SIMPLIFY_BOUNDS_M[0]} et {VECTOR_SIMPLIFY_BOUNDS_M[1]} m.

Ces bornes sont uniquement des garde-fous.

Réponds uniquement en JSON :
{{
  "route_sample_spacing_m": 80.0,
  "polygon_simplify_m": 25.0,
  "justification": "courte justification"
}}
'''

    decision = ask_ollama_json(
        TEXT_MODEL,
        prompt,
    )

    route_spacing = min(
        max(
            float(
                decision[
                    "route_sample_spacing_m"
                ]
            ),
            VECTOR_SAMPLE_BOUNDS_M[0],
        ),
        VECTOR_SAMPLE_BOUNDS_M[1],
    )

    polygon_simplify = min(
        max(
            float(
                decision[
                    "polygon_simplify_m"
                ]
            ),
            VECTOR_SIMPLIFY_BOUNDS_M[0],
        ),
        VECTOR_SIMPLIFY_BOUNDS_M[1],
    )

    return {
        "route_sample_spacing_m": route_spacing,
        "polygon_simplify_m": polygon_simplify,
        "justification": str(
            decision.get(
                "justification",
                "",
            )
        ),
    }


def sample_route_for_agent(spacing_m):
    d = route["distance_m"].to_numpy(dtype=float)
    lat = route["lat"].to_numpy(dtype=float)
    lon = route["lon"].to_numpy(dtype=float)

    samples = np.arange(
        0.0,
        ROUTE_LENGTH_M + spacing_m,
        spacing_m,
    )

    samples = np.unique(
        np.clip(
            samples,
            0.0,
            ROUTE_LENGTH_M,
        )
    )

    return [
        {
            "distance_m": round(
                float(distance),
                1,
            ),
            "lat": round(
                float(
                    np.interp(
                        distance,
                        d,
                        lat,
                    )
                ),
                7,
            ),
            "lon": round(
                float(
                    np.interp(
                        distance,
                        d,
                        lon,
                    )
                ),
                7,
            ),
        }
        for distance in samples
    ]


def build_vector_payload(decision):
    simplified_m = forest_m.simplify(
        decision[
            "polygon_simplify_m"
        ],
        preserve_topology=True,
    )

    simplified_wgs84 = transform(
        to_wgs84.transform,
        simplified_m,
    )

    return {
        "route_length_m": round(
            ROUTE_LENGTH_M,
            1,
        ),
        "route_samples": (
            sample_route_for_agent(
                decision[
                    "route_sample_spacing_m"
                ]
            )
        ),
        # On conserve les tags bruts du contexte,
        # sans annoncer dans le prompt que l'intersection est déjà connue.
        "osm_tags": (
            forest_context[
                "forest_tags"
            ]
        ),
        "osm_geometry": mapping(
            simplified_wgs84
        ),
    }


def run_vector_test():
    decision = (
        choose_vector_representation()
    )

    payload = build_vector_payload(
        decision
    )

    prompt = f'''
Tu es un Agent IA d'interprétation spatiale et sémantique.

Tu reçois :
- une trace d'itinéraire échantillonnée ;
- une géométrie OpenStreetMap ;
- les tags OSM bruts associés.

Détermine :
1. quelle catégorie environnementale générale ces données représentent ;
2. quelles portions de l'itinéraire se trouvent dans cette géométrie.

Tu n'as pas accès au résultat d'une opération géométrique INTERSECT.

Retourne uniquement un JSON :
{{
  "semantic_label": "foret",
  "semantic_confidence": 0.0,
  "semantic_reason": "justification courte",
  "intervals": [
    {{
      "start_m": 1000.0,
      "end_m": 2000.0,
      "confidence": 0.0
    }}
  ]
}}

DONNEES :
{json.dumps(payload, ensure_ascii=False, separators=(",", ":"))}
'''

    response = ask_ollama_json(
        TEXT_MODEL,
        prompt,
    )

    return {
        "representation": decision,
        "semantic_label": response.get(
            "semantic_label",
            "inconnu",
        ),
        "semantic_confidence": float(
            response.get(
                "semantic_confidence",
                0.0,
            )
        ),
        "semantic_reason": str(
            response.get(
                "semantic_reason",
                "",
            )
        ),
        "intervals": (
            normalize_agent_intervals(
                response.get(
                    "intervals",
                    [],
                )
            )
        ),
    }


vector_result = None

if RUN_VECTOR_TEST:
    vector_result = run_vector_test()

    print(
        json.dumps(
            vector_result,
            ensure_ascii=False,
            indent=2,
        )
    )
else:
    print("Test A désactivé.")

# Partie IV — Test B1 : raisonnement spatial visuel

Ce test **n'est pas encore de la vraie reconnaissance visuelle de forêt**.

Python dessine les polygones OSM déjà récupérés, puis la trace et les repères de distance.
Le modèle doit retrouver visuellement les portions où la trace traverse ces surfaces.

La question testée est donc :

> Le modèle sait-il raisonner spatialement à partir d'une représentation graphique ?

In [ ]:
VISION_IMAGE_PATH = (
    OUTPUT_DIR
    / "agent2_testB1_osm_visual.png"
)


def draw_polygon(ax, geometry):
    if geometry is None or geometry.is_empty:
        return

    if geometry.geom_type == "Polygon":
        x, y = geometry.exterior.xy

        ax.fill(
            x,
            y,
            alpha=0.25,
        )

        for interior in geometry.interiors:
            ix, iy = interior.xy

            ax.plot(
                ix,
                iy,
                linewidth=0.5,
            )

    elif geometry.geom_type == "MultiPolygon":
        for polygon in geometry.geoms:
            draw_polygon(
                ax,
                polygon,
            )


def build_visual_osm_context():
    fig, ax = plt.subplots(
        figsize=(10, 10)
    )

    draw_polygon(
        ax,
        forest_m,
    )

    route_line = LineString(
        projected_points
    )

    x_route, y_route = (
        route_line.xy
    )

    ax.plot(
        x_route,
        y_route,
        linewidth=2.5,
        label="Itinéraire",
    )

    xs = np.array(
        [p[0] for p in projected_points]
    )

    ys = np.array(
        [p[1] for p in projected_points]
    )

    markers = np.arange(
        0.0,
        ROUTE_LENGTH_M
        + VISION_MARKER_SPACING_M,
        VISION_MARKER_SPACING_M,
    )

    markers = np.unique(
        np.clip(
            markers,
            0.0,
            ROUTE_LENGTH_M,
        )
    )

    for distance in markers:
        x = float(
            np.interp(
                distance,
                route_distances,
                xs,
            )
        )

        y = float(
            np.interp(
                distance,
                route_distances,
                ys,
            )
        )

        ax.scatter([x], [y], s=18)

        ax.annotate(
            f"{distance/1000:.1f} km",
            (x, y),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )

    ax.set_aspect(
        "equal",
        adjustable="datalim",
    )

    ax.set_title(
        "Test B1 — représentation visuelle OSM"
    )

    ax.legend()
    fig.tight_layout()

    fig.savefig(
        VISION_IMAGE_PATH,
        dpi=VISION_IMAGE_DPI,
        bbox_inches="tight",
    )

    plt.show()

    print("Image :", VISION_IMAGE_PATH)


build_visual_osm_context()

In [ ]:
def image_to_base64(path):
    with path.open("rb") as f:
        return base64.b64encode(
            f.read()
        ).decode("ascii")


def run_visual_osm_test():
    image_b64 = image_to_base64(
        VISION_IMAGE_PATH
    )

    prompt = f'''
Tu reçois une image cartographique abstraite montrant :
- une trace d'itinéraire ;
- des surfaces environnementales ;
- des repères de distance.

Longueur totale de la trace :
{ROUTE_LENGTH_M:.1f} m.

Observe uniquement l'image.

Estime les intervalles de distance où l'itinéraire traverse les surfaces représentées.

Retourne uniquement :
{{
  "intervals": [
    {{
      "start_m": 1000.0,
      "end_m": 2000.0,
      "confidence": 0.0
    }}
  ]
}}
'''

    response = ask_ollama_json(
        VISION_MODEL,
        prompt,
        images_base64=[
            image_b64
        ],
    )

    return {
        "intervals": (
            normalize_agent_intervals(
                response.get(
                    "intervals",
                    [],
                )
            )
        )
    }


visual_osm_result = None

if RUN_VISUAL_OSM_TEST:
    visual_osm_result = (
        run_visual_osm_test()
    )

    print(
        json.dumps(
            visual_osm_result,
            ensure_ascii=False,
            indent=2,
        )
    )
else:
    print(
        "Test B1 désactivé. "
        f"Pour l'activer, installer {VISION_MODEL} "
        "et passer RUN_VISUAL_OSM_TEST=True."
    )

# Partie V — Test B2 à faire ensuite : vraie reconnaissance visuelle de forêt

Le test B2 sera différent de B1.

### Entrée prévue

```text
orthophoto / satellite / photographie aérienne
+
trace GPX
+
repères de distance
```

### Ce que l'image NE devra PAS contenir

```text
pas de polygone OSM forêt déjà dessiné
pas de masque forêt fourni au modèle
pas de tag "forest" dans le prompt
```

### Question scientifique

> Le modèle multimodal sait-il reconnaître visuellement une forêt puis localiser les
> portions de l'itinéraire qui la traversent ?

Ce test nécessite encore de choisir une source d'imagerie reproductible et juridiquement
utilisable. Il est donc volontairement séparé du test B1.

# Partie VI — Évaluation contre la référence SIG

In [ ]:
def intervals_from_result(result):
    if result is None:
        return []

    return [
        (
            float(item["start_m"]),
            float(item["end_m"]),
        )
        for item in result.get(
            "intervals",
            [],
        )
    ]


def interval_union_length(intervals):
    merged = merge_intervals(
        intervals,
        gap_tolerance_m=0.0,
    )

    return sum(
        end - start
        for start, end in merged
    )


def interval_intersection_length(
    intervals_a,
    intervals_b,
):
    a = merge_intervals(
        intervals_a,
        gap_tolerance_m=0.0,
    )

    b = merge_intervals(
        intervals_b,
        gap_tolerance_m=0.0,
    )

    total = 0.0

    for a_start, a_end in a:
        for b_start, b_end in b:
            total += max(
                0.0,
                min(a_end, b_end)
                - max(a_start, b_start),
            )

    return total


def evaluate_intervals(
    predicted,
    reference,
):
    predicted_length = (
        interval_union_length(
            predicted
        )
    )

    reference_length = (
        interval_union_length(
            reference
        )
    )

    inter = (
        interval_intersection_length(
            predicted,
            reference,
        )
    )

    union = (
        predicted_length
        + reference_length
        - inter
    )

    return {
        "predicted_length_m": float(
            predicted_length
        ),
        "reference_length_m": float(
            reference_length
        ),
        "intersection_length_m": float(
            inter
        ),
        "precision_length": float(
            inter / predicted_length
            if predicted_length > 0
            else 0.0
        ),
        "recall_length": float(
            inter / reference_length
            if reference_length > 0
            else 0.0
        ),
        "iou_linear": float(
            inter / union
            if union > 0
            else 1.0
        ),
    }


evaluation = {}

if vector_result is not None:
    evaluation["test_A_vector"] = (
        evaluate_intervals(
            intervals_from_result(
                vector_result
            ),
            reference_intervals,
        )
    )

if visual_osm_result is not None:
    evaluation["test_B1_visual_osm"] = (
        evaluate_intervals(
            intervals_from_result(
                visual_osm_result
            ),
            reference_intervals,
        )
    )

display(
    pd.DataFrame(
        evaluation
    ).T
)

# Partie VII — Export expérimental

Ces fichiers sont volontairement séparés de la sortie officielle de la méthode 0.

Le notebook ne touche jamais à :

```text
terrain_forest_annotations.json
```

In [ ]:
reference_path = (
    OUTPUT_DIR
    / "experiment_reference_intersect.json"
)

vector_path = (
    OUTPUT_DIR
    / "experiment_agent2_vector.json"
)

visual_path = (
    OUTPUT_DIR
    / "experiment_agent2_visual_osm.json"
)

evaluation_path = (
    OUTPUT_DIR
    / "experiment_agent2_evaluation.json"
)

with reference_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        reference_intervals,
        f,
        ensure_ascii=False,
        indent=2,
    )

with vector_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        vector_result,
        f,
        ensure_ascii=False,
        indent=2,
    )

with visual_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        visual_osm_result,
        f,
        ensure_ascii=False,
        indent=2,
    )

with evaluation_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        evaluation,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Exports expérimentaux :")
print("-", reference_path)
print("-", vector_path)
print("-", visual_path)
print("-", evaluation_path)

# Résumé pour le PIR

```text
03bis-A
MÉTHODE 0
OSM + géométrie déterministe
→ résultat stable
→ alimente le notebook 04

03bis-B
BANQUE D'EXPÉRIENCES AGENT IA 2

Référence SIG
INTERSECT

Test A
LLM + vecteurs

Test B1
LLM multimodal + carte abstraite issue des polygones OSM

Test B2 — prochainement
LLM multimodal + vraie imagerie
→ reconnaissance visuelle réelle de forêt
```

Cette séparation permet de continuer le pipeline même si les expériences IA échouent,
tout en gardant un protocole de comparaison reproductible.